# PSEO State Cohort Trends Extraction (Agg 34)

Extracts rows from `pseoe_all.csv` where:
- `agg_level_pseo == 34` (State Cohort Trends)
- `degree_level == 5`
- `cipcode` is one of:
  - `11`, `13`, `51` (output file A)
  - ~~`11`, `13`, `60` (output file B)~~ --> This does not exist as CIP `60` and Deg `5` are mututally exclusive

Reads in chunks to avoid loading the full file into memory.

In [2]:
import pandas as pd

INPUT_FILE = "../Foundational_Data/pseoe_all.csv"
CHUNK_SIZE = 50_000

TARGET_AGG_LEVEL = 34
TARGET_DEGREE_LEVEL = "5"  # normalized (leading zeros removed)

CIP_SET_A = {"01","11", "13", "51"}

OUTPUT_FILE_A = "peoe_agg34_regression.csv"


print(f"Reading '{INPUT_FILE}' in chunks of {CHUNK_SIZE:,} rows...")

Reading '../Foundational_Data/pseoe_all.csv' in chunks of 50,000 rows...


In [3]:
dtype_settings = {
    "agg_level_pseo": int,
    "institution": str,
    "cipcode": str,
    "degree_level": str,
    "grad_cohort": str,
    "geography": str,
    "industry": str,
}


def _normalize_cip(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.zfill(2)


def _normalize_degree_level(s: pd.Series) -> pd.Series:
    # Some files use "05" while others use "5"; normalize to "5"-style.
    return s.astype(str).str.strip().str.lstrip("0").replace({"": "0"})


def extract_to_csv(target_cips: set[str], output_file: str) -> None:
    chunks: list[pd.DataFrame] = []

    for i, chunk in enumerate(
        pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE, dtype=dtype_settings, low_memory=False)
    ):
        chunk["cipcode"] = _normalize_cip(chunk["cipcode"])
        chunk["degree_level"] = _normalize_degree_level(chunk["degree_level"])

        filtered = chunk[
            (chunk["agg_level_pseo"] == TARGET_AGG_LEVEL)
            & (chunk["degree_level"] == TARGET_DEGREE_LEVEL)
            & (chunk["cipcode"].isin(target_cips))
        ]

        if not filtered.empty:
            chunks.append(filtered)

        if (i + 1) % 5 == 0:
            rows_scanned = (i + 1) * CHUNK_SIZE
            print(f"  Scanned ~{rows_scanned:,} rows so far...")

    if chunks:
        df = pd.concat(chunks, ignore_index=True)
        df.to_csv(output_file, index=False)
        print(f"Saved {len(df):,} rows to '{output_file}'")
    else:
        print(f"No matching rows found for CIP set {sorted(target_cips)}. Double-check filters.")


extract_to_csv(CIP_SET_A, OUTPUT_FILE_A)


  Scanned ~250,000 rows so far...
  Scanned ~500,000 rows so far...
Saved 813 rows to 'peoe_agg34_regression.csv'


In [4]:
df = pd.read_csv(OUTPUT_FILE_A, dtype={"cipcode": str, "degree_level": str})
print("=" * 80)
print("File:", OUTPUT_FILE_A)
print("Shape:", df.shape)
print("Unique agg_level_pseo:", sorted(df["agg_level_pseo"].dropna().unique().tolist()))
print("Row counts by cipcode:")
print(df["cipcode"].value_counts().sort_index())
print("Row counts by degree_level:")
print(df["degree_level"].astype(str).value_counts().sort_index())


File: peoe_agg34_regression.csv
Shape: (813, 36)
Unique agg_level_pseo: [34]
Row counts by cipcode:
cipcode
01    185
11    212
13    206
51    210
Name: count, dtype: int64
Row counts by degree_level:
degree_level
5    813
Name: count, dtype: int64


## Partial Listwise Deletion on Earnings Status Flags

Rows are dropped only if **all** earnings status columns are invalid or suppressed. If at least one earnings column has a valid status, the row is retained.

For retained rows with partially invalid data, the **value columns** corresponding to invalid status flags are set to `NaN` so they are automatically excluded from any average or aggregate calculations.

The PSEO data encodes data quality and suppression information in "status" columns. The allowed values we keep are:

**Earnings & graduates earning columns** (`status_y1_earnings`, `status_y1_grads_earn`, `status_y5_earnings`, `status_y5_grads_earn`, `status_y10_earnings`, `status_y10_grads_earn`):
- `1` — Value is valid and released
- `6` — Value is imputed but usable

A row is dropped only when **none** of these six columns contains a valid status. Rows with at least one valid earnings column are kept, but individual earnings value columns with invalid status are nulled out.

In [5]:
EARNINGS_STATUS_COLS = [
    "status_y1_earnings",
    "status_y1_grads_earn",
    "status_y5_earnings",
    "status_y5_grads_earn",
    "status_y10_earnings",
    "status_y10_grads_earn",
]

# Map each status column to the earnings value column it governs
EARNINGS_STATUS_TO_VALUE = {
    "status_y1_earnings":    "y1_earnings",
    "status_y1_grads_earn":  "y1_grads_earn",
    "status_y5_earnings":    "y5_earnings",
    "status_y5_grads_earn":  "y5_grads_earn",
    "status_y10_earnings":   "y10_earnings",
    "status_y10_grads_earn": "y10_grads_earn",
}

VALID_EARNINGS_STATUS = {1, 6}

df_clean = df.copy()
rows_before = len(df_clean)

# Per-column validity mask: True where status is valid
earnings_valid = df_clean[EARNINGS_STATUS_COLS].isin(VALID_EARNINGS_STATUS)

# Drop rows where NO earnings column is valid
has_any_valid = earnings_valid.any(axis=1)
df_clean = df_clean[has_any_valid].reset_index(drop=True)
earnings_valid = earnings_valid.loc[df_clean.index].reset_index(drop=True)

rows_after = len(df_clean)
print(f"Rows before deletion : {rows_before:,}")
print(f"Rows after  deletion : {rows_after:,}")
print(f"Rows dropped         : {rows_before - rows_after:,}")

# Null out value columns whose status is invalid so they are excluded from averages
for status_col, value_col in EARNINGS_STATUS_TO_VALUE.items():
    if value_col in df_clean.columns:
        df_clean.loc[~earnings_valid[status_col], value_col] = float("nan")

nulled_cells = (~earnings_valid).sum().sum()
print(f"Earnings value cells nulled (invalid status) : {nulled_cells:,}")

OUTPUT_FILE_A_CLEAN = OUTPUT_FILE_A.replace(".csv", "_clean.csv")
df_clean.to_csv(OUTPUT_FILE_A_CLEAN, index=False)
print(f"\nSaved cleaned dataset to '{OUTPUT_FILE_A_CLEAN}'")

Rows before deletion : 813
Rows after  deletion : 763
Rows dropped         : 50
Earnings value cells nulled (invalid status) : 1,166

Saved cleaned dataset to 'peoe_agg34_regression_clean.csv'


## Add State Name Column

The `institution` column in the cleaned dataset contains a numeric FIPS code identifying the state (e.g., `37` for North Carolina). The file `label_fipsnum.csv` maps these codes to human-readable state names via its `geography` column, which stores the same codes zero-padded to two digits (e.g., `"37"`).

To make the dataset easier to work with, we:
1. Zero-pad `institution` to two digits to match the format in `label_fipsnum.csv`
2. Merge on that key to bring in the `label` column (the state name)
3. Rename `label` → `state_name` and insert it as the second column (after `agg_level_pseo`) for readability
4. Save the result back to the clean CSV

In [6]:
df_clean = pd.read_csv(OUTPUT_FILE_A_CLEAN, dtype={"institution": str})

# Load FIPS → state name mapping
fips = pd.read_csv("../Labels/label_fipsnum.csv", dtype={"geography": str})
fips = fips.rename(columns={"label": "state_name"})[["geography", "state_name"]]

# Zero-pad institution to 2 digits to match fips geography format
df_clean["institution_fips"] = df_clean["institution"].str.zfill(2)

# Merge to bring in state_name
df_clean = df_clean.merge(fips, left_on="institution_fips", right_on="geography", how="left")
df_clean = df_clean.drop(columns=["institution_fips", "geography_y"], errors="ignore")

# Rename geography_x → geography if the merge created a suffix
if "geography_x" in df_clean.columns:
    df_clean = df_clean.rename(columns={"geography_x": "geography"})

# Insert state_name as the second column (after agg_level_pseo)
cols = df_clean.columns.tolist()
cols.remove("state_name")
insert_after = cols.index("agg_level_pseo") + 1
cols.insert(insert_after, "state_name")
df_clean = df_clean[cols]

df_clean.to_csv(OUTPUT_FILE_A_CLEAN, index=False)
print(f"state_name sample:\n{df_clean[['institution', 'state_name']].drop_duplicates().sort_values('institution').to_string(index=False)}")
print(f"\nRows with missing state_name: {df_clean['state_name'].isna().sum()}")

state_name sample:
institution           state_name
          1              Alabama
         11 District of Columbia
         13              Georgia
         15               Hawaii
         16                Idaho
         17             Illinois
         18              Indiana
         19                 Iowa
         22            Louisiana
         23                Maine
         25        Massachusetts
         26             Michigan
         27            Minnesota
         29             Missouri
         30              Montana
         36             New York
         37       North Carolina
         39                 Ohio
          4              Arizona
         40             Oklahoma
         41               Oregon
         42         Pennsylvania
         44         Rhode Island
         45       South Carolina
         46         South Dakota
         48                Texas
         49                 Utah
         51             Virginia
         54        West 

## Cohort Label

The `grad_cohort` column stores the starting year of a 3-year cohort window. For example, a `grad_cohort` of `2001` covers graduates from 2001, 2002, and 2003. A new string column `grad_cohort_label` is created in the format `"YYYY-YYYY+2"` (e.g., `"2001-2003"`) to make cohort-wise visualizations more interpretable on axes and legends.

In [7]:
df_clean = pd.read_csv(OUTPUT_FILE_A_CLEAN)

# Add grad_cohort_label: each cohort spans 3 years (e.g., 2001 → "2001-2003")
df_clean["grad_cohort_label"] = df_clean["grad_cohort"].apply(
    lambda y: f"{y}-{y + 2}"
)

df_clean.to_csv(OUTPUT_FILE_A_CLEAN, index=False)
print(f"Columns: {df_clean.columns.tolist()}")
print(f"\nSample cohort labels:\n{df_clean[['grad_cohort', 'grad_cohort_label']].drop_duplicates().sort_values('grad_cohort').to_string(index=False)}")

Columns: ['agg_level_pseo', 'state_name', 'inst_level', 'institution', 'degree_level', 'cip_level', 'cipcode', 'grad_cohort', 'grad_cohort_years', 'geo_level', 'geography', 'ind_level', 'industry', 'y1_p25_earnings', 'y1_p50_earnings', 'y1_p75_earnings', 'y1_grads_earn', 'y5_p25_earnings', 'y5_p50_earnings', 'y5_p75_earnings', 'y5_grads_earn', 'y10_p25_earnings', 'y10_p50_earnings', 'y10_p75_earnings', 'y10_grads_earn', 'y1_ipeds_count', 'y5_ipeds_count', 'y10_ipeds_count', 'status_y1_earnings', 'status_y1_grads_earn', 'status_y5_earnings', 'status_y5_grads_earn', 'status_y10_earnings', 'status_y10_grads_earn', 'status_y1_ipeds_count', 'status_y5_ipeds_count', 'status_y10_ipeds_count', 'grad_cohort_label']

Sample cohort labels:
 grad_cohort grad_cohort_label
        2001         2001-2003
        2004         2004-2006
        2007         2007-2009
        2010         2010-2012
        2013         2013-2015
        2016         2016-2018
        2019         2019-2021


In [8]:
df_clean = pd.read_csv(OUTPUT_FILE_A_CLEAN)
print(df_clean.shape)
print("="*80)
print(df_clean.columns)

(763, 38)
Index(['agg_level_pseo', 'state_name', 'inst_level', 'institution',
       'degree_level', 'cip_level', 'cipcode', 'grad_cohort',
       'grad_cohort_years', 'geo_level', 'geography', 'ind_level', 'industry',
       'y1_p25_earnings', 'y1_p50_earnings', 'y1_p75_earnings',
       'y1_grads_earn', 'y5_p25_earnings', 'y5_p50_earnings',
       'y5_p75_earnings', 'y5_grads_earn', 'y10_p25_earnings',
       'y10_p50_earnings', 'y10_p75_earnings', 'y10_grads_earn',
       'y1_ipeds_count', 'y5_ipeds_count', 'y10_ipeds_count',
       'status_y1_earnings', 'status_y1_grads_earn', 'status_y5_earnings',
       'status_y5_grads_earn', 'status_y10_earnings', 'status_y10_grads_earn',
       'status_y1_ipeds_count', 'status_y5_ipeds_count',
       'status_y10_ipeds_count', 'grad_cohort_label'],
      dtype='str')


In [13]:
import statsmodels.formula.api as smf
import os

# Ensure clean types
df_clean['cipcode'] = df_clean['cipcode'].astype(str).str.zfill(2)
df_clean['geography'] = df_clean['geography'].astype(str)
df_clean['grad_cohort'] = df_clean['grad_cohort'].astype(str)

# Create the Fixed Effects "Vortex"
df_clean['state_cohort'] = df_clean['geography'] + "_" + df_clean['grad_cohort']

years = ['y1', 'y5', 'y10']
percentiles = ['p25', 'p50', 'p75']
os.makedirs('appendix_tables', exist_ok=True)

coef_data = []

for y in years:
    for p in percentiles:
        target = f"{y}_{p}_earnings"
        
        # areg-equivalent: Regressing on CIP while absorbing State-Cohort
        model = smf.ols(f'{target} ~ C(cipcode, Treatment("01")) + C(state_cohort)', data=df_clean).fit()
        
        # 1. Export HTML Table for Appendix
        with open(f'appendix_tables/table_{target}.html', 'w') as f:
            f.write(model.summary().as_html())
            
        # 2. Store Coefficients for Dashboard
        betas = model.params.filter(like='cipcode')
        for cip_raw, val in betas.items():
            cip_label = cip_raw.split('[T.')[1].replace(']', '')
            coef_data.append({
                'year_after': y,
                'percentile': p,
                'cip': cip_label,
                'premium': val,
                'p_value': model.pvalues[cip_raw]
            })

# Save for Streamlit
pd.DataFrame(coef_data).to_csv('model_coefficients.csv', index=False)

/Users/anubhavdhungana/Desktop/BSc. Computer Science/Symposium_Dashboard/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1966: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(eigvals[0]/eigvals[-1])
/Users/anubhavdhungana/Desktop/BSc. Computer Science/Symposium_Dashboard/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1966: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(eigvals[0]/eigvals[-1])
/Users/anubhavdhungana/Desktop/BSc. Computer Science/Symposium_Dashboard/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1966: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(eigvals[0]/eigvals[-1])
/Users/anubhavdhungana/Desktop/BSc. Computer Science/Symposium_Dashboard/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1966: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(eigvals[